In [3]:
# Block 1 (Updated): Free / API-Key-Free Model Setup (Using HuggingFace / Transformers)
!pip install -qU langgraph langchain-core langchain-community transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [5]:
# Block 2: Imports, Free Model Setup (HuggingFace Pipeline), and State Definition
import operator
from typing import TypedDict, Annotated
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langgraph.graph import StateGraph, START, END

# Free Open-Source Model (No API Key required)
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.7
)

llm = HuggingFacePipeline(pipeline=pipe)

# State Definition:
# `operator.add` annotated list ensures that results from parallel nodes
# are merged into a single list (Fan-in) rather than overwriting each other.
class State(TypedDict):
    topic: str
    results: Annotated[list, operator.add]

/tmp/ipykernel_1672/1602315115.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFacePipeline


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_1672/1602315115.py:21: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [10]:
# Block 3 (Fixed Prompting Format):

def node_pros(state: State):
    """Parallel Node 1: Pros analyze karega"""
    formatted_prompt = f"<|user|>\nList 2 short advantages of {state['topic']}.\n<|assistant|>\n"
    response = llm.invoke(formatted_prompt)
    # Output cleaning to remove prompt text if repeated
    cleaned = response.replace(formatted_prompt, "").strip()
    return {"results": [f"--- PROS ---\n{cleaned}"]}

def node_cons(state: State):
    """Parallel Node 2: Cons analyze karega"""
    formatted_prompt = f"<|user|>\nList 2 short disadvantages of {state['topic']}.\n<|assistant|>\n"
    response = llm.invoke(formatted_prompt)
    cleaned = response.replace(formatted_prompt, "").strip()
    return {"results": [f"--- CONS ---\n{cleaned}"]}

def aggregator_node(state: State):
    """Fan-in Node: Summarize and return final state"""
    return state

In [11]:
# Block 4: Build, Connect, and Compile the Parallel Graph
builder = StateGraph(State)

# Nodes add karein
builder.add_node("pros_node", node_pros)
builder.add_node("cons_node", node_cons)
builder.add_node("aggregator", aggregator_node)

# Parallel Branching (Fan-out):
# START se dono nodes ko parallel connect karein
builder.add_edge(START, "pros_node")
builder.add_edge(START, "cons_node")

# Parallel Joining (Fan-in):
# Dono nodes ka output aggregator node par aakar milega
builder.add_edge("pros_node", "aggregator")
builder.add_edge("cons_node", "aggregator")
builder.add_edge("aggregator", END)

# Graph compile karein
graph = builder.compile()

In [13]:
# Block 5 (Fixed Single Execution):
inputs = {"topic": "Artificial Intelligence"}

# Invoke graph ONCE and save output directly
output = graph.invoke(inputs)

print("=== FINAL AGGREGATED RESULTS ===")
for idx, res in enumerate(output["results"], 1):
    print(f"\nResult {idx}:\n{res}")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== FINAL AGGREGATED RESULTS ===

Result 1:
--- CONS ---
1. Rising Costs: As AI algorithms become more advanced, the resources required to build and maintain them become more costly. Therefore, companies may need to invest more in AI technology to make it affordable for their business operations.

2. Privacy Concerns: AI systems can be highly sensitive to data privacy. This can lead to issues such as data breaches, misuse of personal information, and reputational damage for companies.

3

Result 2:
--- PROS ---
1. Improved decision-making: AI algorithms can analyze data more efficiently and provide more accurate predictions, which can lead to more efficient and effective decision-making.

2. Increased productivity: AI can automate repetitive and complex tasks, freeing up employees to focus on more complex and creative work. This can lead to increased productivity and better results.

3. Enhanced customer experience: AI can personalize the customer experience by understanding

Result 3: